# Type 7 RAG Evaluation

This notebook is the professor-facing evaluation surface for the Type 7 RAG workflow.

It assumes the canonical retrieval pipeline has already produced `results/runs/rrf_bge_m3.csv`, and that the Type 7 RAG scripts have produced the query, context, answer, and evaluation artifacts described in `docs/rerun_pipeline.md`.

You can use it in two ways:

1. inspect existing evaluation outputs that were produced by the scripts
2. optionally rerun the backend evaluation script from inside the notebook


In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "data" / "raw_data").exists():
    for parent in Path.cwd().parents:
        if (parent / "data" / "raw_data").exists():
            ROOT = parent
            break

RUN_TAG = "current"
RUN_BACKEND_EVALUATION = False
EVALUATOR_MODEL = "gpt-oss:20b"
OLLAMA_URL = "http://localhost:11434"

QUERIES_PATH = ROOT / "data" / "queries" / "type7_queries.json"
CONTEXT_PATH = ROOT / "results" / "rag" / f"context_{RUN_TAG}.jsonl"
ANSWERS_PATH = ROOT / "results" / "rag" / f"answers_{RUN_TAG}.jsonl"
OUTPUT_DIR = ROOT / "results" / "rag" / "evaluation" / RUN_TAG
PER_QUERY_PATH = OUTPUT_DIR / "per_query.csv"
SUMMARY_PATH = OUTPUT_DIR / "summary.json"

print(f"ROOT: {ROOT}")
print(f"RUN_TAG: {RUN_TAG}")
print(f"RUN_BACKEND_EVALUATION: {RUN_BACKEND_EVALUATION}")


In [ ]:
if RUN_BACKEND_EVALUATION:
    cmd = [
        sys.executable,
        str(ROOT / "scripts" / "rag" / "evaluate_answers.py"),
        "--answers",
        str(ANSWERS_PATH),
        "--context",
        str(CONTEXT_PATH),
        "--queries",
        str(QUERIES_PATH),
        "--output-dir",
        str(OUTPUT_DIR),
        "--evaluator-model",
        EVALUATOR_MODEL,
        "--ollama-url",
        OLLAMA_URL,
    ]
    print("Running backend evaluation...\n")
    subprocess.run(cmd, check=True, cwd=ROOT)
else:
    print("Skipping backend evaluation. Set RUN_BACKEND_EVALUATION = True to rerun it from the notebook.")


In [ ]:
required_paths = [QUERIES_PATH, PER_QUERY_PATH, SUMMARY_PATH]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing required files for notebook review:\n- " + "\n- ".join(missing)
    )

with QUERIES_PATH.open("r", encoding="utf-8") as handle:
    queries_payload = json.load(handle)

queries_df = pd.DataFrame(queries_payload.get("queries", []))
per_query_df = pd.read_csv(PER_QUERY_PATH)
with SUMMARY_PATH.open("r", encoding="utf-8") as handle:
    summary = json.load(handle)

print(f"Type 7 queries: {len(queries_df)}")
print(f"Evaluated queries: {len(per_query_df)}")
print("\nAggregate summary:")
pd.DataFrame([summary])


In [ ]:
display_cols = [
    "query_id",
    "faithfulness",
    "answer_relevancy",
    "context_precision",
    "context_recall",
    "pseudo_ground_truth",
]
available = [col for col in display_cols if col in per_query_df.columns]
per_query_df[available].sort_values("query_id").head(20)


In [ ]:
context_rows = []
with CONTEXT_PATH.open("r", encoding="utf-8") as handle:
    for line in handle:
        if line.strip():
            context_rows.append(json.loads(line))
context_df = pd.DataFrame(context_rows)

answer_rows = []
with ANSWERS_PATH.open("r", encoding="utf-8") as handle:
    for line in handle:
        if line.strip():
            answer_rows.append(json.loads(line))
answers_df = pd.DataFrame(answer_rows)

sample_qid = per_query_df.sort_values("query_id").iloc[0]["query_id"]
print(f"Sample query: {sample_qid}\n")
display(queries_df[queries_df["query_id"] == sample_qid][["query_id", "source_query_id", "source_type", "query_text", "query_notes"]])
display(answers_df[answers_df["query_id"] == sample_qid][["query_id", "model_name", "prompt_version", "answer"]])
display(context_df[context_df["query_id"] == sample_qid][["query_id", "doc_id", "rank", "caption", "schema", "meta_programId"]].head(10))
